# BirdCLEF+ 2026 ONNX Distilled SED Submission

Inference-only notebook for the next submission lane. This notebook uses public distilled SED ONNX folds and writes a Kaggle-compatible `submission.csv` with all 234 target columns. It intentionally avoids training, TensorFlow Perch inference, and debug fallbacks that can hide hidden-test runtime risk.

## Setup

Use fixed Kaggle input roots for reproducibility. The only model dependency here is the distilled SED ONNX fold artifact; the Perch ONNX path is used only as a convenient source for the offline ONNX Runtime wheel.

In [3]:
from pathlib import Path
import os
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


class CFG:
    """Configuration for distilled SED ONNX submission inference."""

    seed = 42
    data_root = Path("/kaggle/input/competitions/birdclef-2026")
    sed_model_dir = Path(
        "/kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public"
    )
    onnx_wheel_root = Path(
        "/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx"
    )
    output_path = Path("/kaggle/working/submission.csv")

    sample_rate = 32_000
    clip_seconds = 5
    soundscape_seconds = 60
    n_windows = soundscape_seconds // clip_seconds

    n_mels = 256
    n_fft = 2048
    hop_length = 512
    fmin = 20
    fmax = 16_000
    top_db = 80

    fold_pattern = "sed_fold*.onnx"
    onnx_threads = 4
    smooth_sigma = 0.75


np.random.seed(CFG.seed)
print("Competition root:", CFG.data_root)
print("SED model root:", CFG.sed_model_dir)


Competition root: /kaggle/input/competitions/birdclef-2026
SED model root: /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public


In [4]:
def install_onnxruntime_if_needed():
    """Import ONNX Runtime, installing the attached wheel if needed.

    Returns:
        module: Imported `onnxruntime` module.

    Raises:
        FileNotFoundError: If ONNX Runtime is unavailable and no offline wheel
            is attached under `/kaggle/input`.
    """
    try:
        import onnxruntime as ort

        return ort
    except ImportError:
        wheels = sorted(CFG.onnx_wheel_root.glob("onnxruntime-*.whl"))
        if not wheels:
            wheels = sorted(Path("/kaggle/input").glob("**/onnxruntime-*.whl"))
        if not wheels:
            raise FileNotFoundError(
                "No offline onnxruntime wheel found under /kaggle/input."
            )
        wheel = wheels[0]
        print("Installing ONNX Runtime from", wheel)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                str(wheel),
            ],
            check=True,
        )
        import onnxruntime as ort

        return ort


ort = install_onnxruntime_if_needed()
import librosa
from scipy.ndimage import gaussian_filter1d

print("ONNX Runtime:", ort.__version__)


Installing ONNX Runtime from /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
ONNX Runtime: 1.24.4


## Load Submission Contract And Models

The submission column order always comes from `sample_submission.csv`. This keeps the notebook aligned with the competition output contract even if model internals use a different ordering.

In [5]:
sample_submission = pd.read_csv(CFG.data_root / "sample_submission.csv")
target_columns = [c for c in sample_submission.columns if c != "row_id"]
num_classes = len(target_columns)
print("Sample submission:", sample_submission.shape)
print("Target columns:", num_classes)


def find_sed_folds():
    """Find distilled SED ONNX fold files from Kaggle inputs.

    Returns:
        list[Path]: Sorted ONNX fold paths.

    Raises:
        FileNotFoundError: If no fold files are found under `/kaggle/input`.
    """
    model_dir = CFG.sed_model_dir
    folds = sorted(model_dir.glob(CFG.fold_pattern))
    if not folds:
        folds = sorted(Path("/kaggle/input").glob(f"**/{CFG.fold_pattern}"))
    if not folds:
        raise FileNotFoundError(
            f"No SED ONNX folds matching {CFG.fold_pattern!r} found under /kaggle/input."
        )
    return folds


sed_folds = find_sed_folds()
print("SED folds:")
for path in sed_folds:
    print(" -", path)


Sample submission: (3, 235)
Target columns: 234
SED folds:
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold0.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold1.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold2.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold3.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold4.onnx


In [6]:
def make_session(path):
    """Create an optimized CPU ONNX Runtime session.

    Args:
        path (Path): ONNX model file path.

    Returns:
        onnxruntime.InferenceSession: Loaded CPU inference session.
    """
    options = ort.SessionOptions()
    options.intra_op_num_threads = CFG.onnx_threads
    options.inter_op_num_threads = 1
    options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )
    return ort.InferenceSession(
        str(path), sess_options=options, providers=["CPUExecutionProvider"]
    )


sessions = [make_session(path) for path in sed_folds]
input_names = [session.get_inputs()[0].name for session in sessions]
print(f"Loaded {len(sessions)} ONNX fold sessions")
print(
    "Input:",
    sessions[0].get_inputs()[0].name,
    sessions[0].get_inputs()[0].shape,
)
print("Outputs:", [(out.name, out.shape) for out in sessions[0].get_outputs()])


Loaded 5 ONNX fold sessions
Input: mel ['batch', 1, 256, 313]
Outputs: [('clip_logits', ['batch', 234]), ('framewise_logits', ['batch', 'batch', 234])]


## Inference Helpers

Each 60-second soundscape is padded or trimmed to exactly 60 seconds, split into 12 contiguous 5-second windows, converted to log-mel features, then scored by every fold.

In [7]:
def sigmoid(x):
    """Apply a numerically stable sigmoid transform.

    Args:
        x (np.ndarray): Logit array.

    Returns:
        np.ndarray: Probability array with the same shape as `x`.
    """
    x = np.clip(x, -40, 40)
    return 1.0 / (1.0 + np.exp(-x))


def load_soundscape_chunks(audio_path):
    """Load one soundscape and split it into fixed 5-second chunks.

    Args:
        audio_path (Path): Path to a 60-second OGG soundscape.

    Returns:
        np.ndarray: Array shaped `(n_windows, samples_per_window)`.
    """
    audio, _ = librosa.load(audio_path, sr=CFG.sample_rate, mono=True)
    expected = CFG.soundscape_seconds * CFG.sample_rate
    if len(audio) < expected:
        audio = np.pad(audio, (0, expected - len(audio)))
    else:
        audio = audio[:expected]
    return audio.reshape(CFG.n_windows, CFG.clip_seconds * CFG.sample_rate)


def chunks_to_logmel(chunks):
    """Convert waveform chunks into normalized log-mel tensors.

    Args:
        chunks (np.ndarray): Waveform chunks from `load_soundscape_chunks`.

    Returns:
        np.ndarray: Float32 tensor shaped `(n_windows, 1, n_mels, frames)`.
    """
    features = []
    for chunk in chunks:
        mel = librosa.feature.melspectrogram(
            y=chunk,
            sr=CFG.sample_rate,
            n_fft=CFG.n_fft,
            hop_length=CFG.hop_length,
            n_mels=CFG.n_mels,
            fmin=CFG.fmin,
            fmax=CFG.fmax,
            power=2.0,
        )
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=CFG.top_db)
        mel_db = (mel_db + CFG.top_db) / CFG.top_db
        features.append(mel_db.astype(np.float32))
    return np.expand_dims(np.stack(features, axis=0), axis=1)


def predict_soundscape(audio_path):
    """Predict target probabilities for one soundscape file.

    Args:
        audio_path (Path): Path to one hidden test soundscape.

    Returns:
        tuple[list[str], np.ndarray]: Row IDs and probability matrix for the
            12 contiguous 5-second windows.
    """
    chunks = load_soundscape_chunks(audio_path)
    batch = chunks_to_logmel(chunks)

    logits_sum = np.zeros((CFG.n_windows, num_classes), dtype=np.float32)
    for session, input_name in zip(sessions, input_names):
        outputs = session.run(None, {input_name: batch})
        clip_logits = outputs[0]
        frame_logits = outputs[1]
        frame_max = frame_logits.max(axis=1)
        logits_sum += 0.5 * clip_logits + 0.5 * frame_max

    logits = logits_sum / len(sessions)
    logits = gaussian_filter1d(
        logits, sigma=CFG.smooth_sigma, axis=0, mode="nearest"
    )
    probs = sigmoid(logits).astype(np.float32)

    stem = Path(audio_path).stem
    row_ids = [
        f"{stem}_{(i + 1) * CFG.clip_seconds}" for i in range(CFG.n_windows)
    ]
    return row_ids, probs


## Write Submission

On the public notebook preview, Kaggle may expose only the three-row sample submission and no `test_soundscapes`. In that case this notebook writes the untouched sample file. During real scoring, hidden soundscapes are mounted and every expected row must be predicted.

In [8]:
def list_test_soundscapes():
    """List hidden test soundscapes when Kaggle mounts them.

    Returns:
        list[Path]: Sorted OGG files from `test_soundscapes`, or an empty list
            during public dry runs.
    """
    test_dir = CFG.data_root / "test_soundscapes"
    if not test_dir.exists():
        return []
    return sorted(test_dir.glob("*.ogg"))


def build_submission():
    """Score test soundscapes and write a Kaggle submission file.

    Returns:
        pd.DataFrame: Submission frame written to `CFG.output_path`.

    Raises:
        ValueError: If hidden scoring expects rows that were not predicted.
    """
    audio_paths = list_test_soundscapes()
    if not audio_paths:
        print(
            "No test soundscapes found. Writing sample submission for public dry run."
        )
        submission = sample_submission.copy()
        submission.to_csv(CFG.output_path, index=False)
        return submission

    print(f"Scoring {len(audio_paths)} soundscape files")
    start = time.time()
    rows = []
    values = []

    for idx, audio_path in enumerate(audio_paths, start=1):
        row_ids, probs = predict_soundscape(audio_path)
        rows.extend(row_ids)
        values.append(probs)
        if idx == 1 or idx % 25 == 0 or idx == len(audio_paths):
            elapsed = time.time() - start
            print(f"{idx}/{len(audio_paths)} files scored in {elapsed:.1f}s")

    pred_df = pd.DataFrame(np.vstack(values), columns=target_columns)
    pred_df.insert(0, "row_id", rows)
    pred_df = pred_df.drop_duplicates("row_id").set_index("row_id")

    missing = sorted(set(sample_submission["row_id"]) - set(pred_df.index))
    if missing:
        raise ValueError(
            f"Missing predictions for {len(missing)} row_ids. Example: {missing[:3]}"
        )

    submission = sample_submission[["row_id"]].copy()
    submission[target_columns] = pred_df.loc[
        submission["row_id"], target_columns
    ].to_numpy()
    submission.to_csv(CFG.output_path, index=False)
    print("Saved", CFG.output_path, submission.shape)
    return submission


submission = build_submission()
display(submission.head())
print(submission.shape)


No test soundscapes found. Writing sample submission for public dry run.


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


(3, 235)
